# Codebook Suite: OPT + Llama (W4A4)

For each model this **solves** the constrained-optimal 1/16-lattice codebook on that model's *own* post-Hadamard activations (a short calibration pass → exact enumeration; `solved_opt`, and `solved_pop2` under a popcount≤2 shift-and-add constraint), and evaluates it alongside four fixed reference codebooks. The solved-per-model levels are printed so you can see whether the optimum is universal (matches the global `{0,2,4,6,8,10,13,16}`) across families and scale.

Reference codebooks:

| codebook | levels ×16 | note |
|---|---|---|
| `exact_GF4` | continuous | hand-tuned Gaussian-quantile |
| `round_Q1.4` | {0,1,3,5,6,8,11,16} | GF4 rounded to 1/16 grid |
| `opt_Q1.4` | {0,2,4,6,8,10,13,16} | **constrained optimum** |
| `opt_pop2` | {0,2,4,6,8,10,12,16} | + popcount≤2 (shift-and-add) |

**W4A4**: weights → NVFP4/E2M1 (E4M3 block scale, our microscaling), activations → the codebook, both under a blockwise Hadamard. WikiText-2 perplexity; `dPPL(exact)` is vs hand-tuned GF4 (the honest read is **iso-accuracy** — the win is 61% cheaper decode, not lower PPL).

Suite: the **OPT ladder (125m–13B)** plus **Mistral-7B** and **Qwen2.5-7B/14B** — three families, all fitting a 40 GB Colab A100. `PURGE_CACHE=True` deletes each model's download right after it's scored, so **peak disk = the largest single model (~28 GB), not the sum** — this is what keeps Colab from filling up mid-run. Results write incrementally, so a later OOM/disk crash never loses finished rows. Set `WEIGHTS='fp16'` for the A-only ablation. **Runtime → Change runtime type → A100 GPU, High-RAM.**

In [ ]:
!pip install -q transformers datasets accelerate scipy sentencepiece
# OPT / Mistral / Qwen are open -- no HF login needed. (Only gated models like Llama need:
#   from huggingface_hub import login; login() )

In [ ]:
import math, csv, gc, itertools
import numpy as np
import torch, torch.nn as nn
from scipy.linalg import hadamard as scipy_hadamard

GF4_LEVEL = np.array([0.0, 0.0796082, 0.1737177, 0.2828685,
                      0.3952704, 0.5250730, 0.6961928, 1.0], dtype=np.float64)
BLOCK = 32
CLIP_RATIO = 2.5

# The four candidate codebooks (levels x 16 on the 1/16 grid).
CODEBOOKS = {
    "exact_GF4":  GF4_LEVEL,
    "round_Q1.4": np.array([0, 1, 3, 5, 6, 8, 11, 16]) / 16,   # GF4 rounded
    "opt_Q1.4":   np.array([0, 2, 4, 6, 8, 10, 13, 16]) / 16,  # constrained optimum
    "opt_pop2":   np.array([0, 2, 4, 6, 8, 10, 12, 16]) / 16,  # + popcount<=2 (shift-add)
}

def quant_e4m3(scale):
    s = scale.clamp(min=2.0 ** -9, max=448.0)
    e = torch.floor(torch.log2(s))
    m = torch.round((s / torch.exp2(e) - 1.0) * 8.0) / 8.0
    return (1.0 + m) * torch.exp2(e)
_SCALE_MODE = "fp16"

# NVFP4 weight quant (E2M1 elements + our E4M3 block scale) -> the real W4A4 scheme
E2M1_MAG = np.array([0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0], dtype=np.float32)
WBLOCK = 16
QUANT_WEIGHTS_NVFP4 = True   # True = W4A4 (NVFP4 weights); False = A-only (fp16 weights)
_E2M1_LV = _E2M1_THR = None
def _ensure_e2m1(device):
    global _E2M1_LV, _E2M1_THR
    if _E2M1_LV is None or _E2M1_LV.device != device:
        _E2M1_LV = torch.tensor(E2M1_MAG, device=device)
        _E2M1_THR = torch.tensor((E2M1_MAG[:-1] + E2M1_MAG[1:]) / 2.0, device=device)
def nvfp4_weight_quant(W, block=WBLOCK):
    dev = W.device; _ensure_e2m1(dev); shp = W.shape
    if shp[1] % block != 0:
        block = 32 if shp[1] % 32 == 0 else BLOCK
    wb = W.reshape(-1, block).float()
    amax = wb.abs().amax(dim=1, keepdim=True).clamp_min(1e-8)
    scale = quant_e4m3(amax / 6.0)
    wn = wb / scale
    idx = torch.bucketize(wn.abs().clamp(0.0, 6.0), _E2M1_THR)
    return (_E2M1_LV[idx] * torch.sign(wn) * scale).reshape(shp).to(W.dtype)

_LEVELS = _THR = None
def set_codebook(levels_np, device):
    global _LEVELS, _THR
    lv = np.sort(levels_np).astype(np.float32)
    thr = np.array([(lv[i] + lv[i + 1]) / 2 for i in range(len(lv) - 1)], dtype=np.float32)
    _LEVELS = torch.tensor(lv, device=device); _THR = torch.tensor(thr, device=device)

_H = None
def hmat(device, dtype):
    global _H
    if _H is None or _H.device != device:
        _H = torch.tensor(scipy_hadamard(BLOCK).astype(np.float32) / math.sqrt(BLOCK), device=device)
    return _H.to(dtype)
def rotate(x, signs):
    F_ = x.shape[-1]; H = hmat(x.device, x.dtype)
    xr = (x * signs).reshape(*x.shape[:-1], F_ // BLOCK, BLOCK) @ H
    return xr.reshape(*x.shape[:-1], F_)

def gf4_quant(x):
    shp = x.shape
    xb = x.reshape(-1, BLOCK).float()
    rms = xb.pow(2).mean(-1, keepdim=True).add(1e-12).sqrt()
    scale = rms * CLIP_RATIO
    if _SCALE_MODE == "e4m3": scale = quant_e4m3(scale)
    xn = xb / scale
    thr = _THR.to(xb.device); lv = _LEVELS.to(xb.device)   # device-safe (multi-GPU)
    idx = torch.bucketize(xn.abs().clamp(0.0, 1.0), thr)
    return (lv[idx] * torch.sign(xn) * scale).reshape(shp).to(x.dtype)

# --- constrained-optimal codebook solved ON THIS MODEL's own activations ------
_CALIB = None    # when a list, pre-hook records normalized mags instead of quantizing
def solve_constrained_codebook(mags, grid=16, popcount=None):
    m = np.sort(np.asarray(mags, dtype=np.float64)); w = np.full(m.shape, 1.0/len(m))
    cP = np.concatenate([[0.0], np.cumsum(w)]); cM = np.concatenate([[0.0], np.cumsum(w*m)])
    cM2 = np.concatenate([[0.0], np.cumsum(w*m*m)])
    def cum(x): i = np.searchsorted(m, x, side="right"); return cP[i], cM[i], cM2[i]
    def dist(ks):
        lv = np.array(ks, dtype=np.float64)/grid
        b = np.concatenate([[0.0], (lv[:-1]+lv[1:])/2, [1.0+1e-9]]); d = 0.0
        for t, l in enumerate(lv):
            P0,M0,M20 = cum(b[t]); P1,M1,M21 = cum(b[t+1])
            d += (M21-M20) - 2*l*(M1-M0) + l*l*(P1-P0)
        return d
    pool = [k for k in range(1, grid) if popcount is None or bin(k).count("1") <= popcount]
    best, bd = None, float("inf")
    for c in itertools.combinations(pool, 6):
        ks = (0,)+c+(grid,); d = dist(ks)
        if d < bd: bd, best = d, ks
    return np.array(best, dtype=np.float64)/grid, best
def collect_mags(model, windows, n_windows=2, max_samples=1_500_000):
    global _CALIB
    _CALIB = []; perplexity(model, windows[:n_windows]); mags = np.concatenate(_CALIB); _CALIB = None
    if mags.shape[0] > max_samples:
        mags = np.random.default_rng(0).choice(mags, max_samples, replace=False)
    return mags

def install_hooks(model, seed=0):
    g = torch.Generator().manual_seed(seed); n = 0
    for name, m in model.named_modules():
        if isinstance(m, nn.Linear) and m.in_features % BLOCK == 0 and "lm_head" not in name:
            dev = m.weight.device
            signs = (torch.randint(0, 2, (m.in_features,), generator=g).float() * 2 - 1).to(dev)
            m.register_buffer("_gf4_signs", signs, persistent=False)
            with torch.no_grad():
                Wr = rotate(m.weight.data.float(), signs)
                if QUANT_WEIGHTS_NVFP4: Wr = nvfp4_weight_quant(Wr)
                m.weight.data.copy_(Wr.to(m.weight.dtype))
            def pre_hook(mod, inp):
                x = inp[0]; signs = mod._gf4_signs.to(x.device)   # device-safe
                xr = rotate(x.float(), signs)
                if _CALIB is not None:                            # calibration: record mags
                    xb = xr.reshape(-1, BLOCK)
                    rms = xb.pow(2).mean(-1, keepdim=True).add(1e-12).sqrt()
                    mg = (xb/(rms*CLIP_RATIO)).abs().clamp(0.0, 1.0).reshape(-1)
                    sel = torch.randint(0, mg.numel(), (min(4096, mg.numel()),), device=mg.device)
                    _CALIB.append(mg[sel].detach().float().cpu().numpy()); return inp
                return (gf4_quant(xr).to(x.dtype),) + inp[1:]
            m.register_forward_pre_hook(pre_hook); n += 1
    return n

def load_windows(tok, seqlen, n):
    from datasets import load_dataset
    ds = None                                    # parquet-hosted mirror -> no loading script
    for repo in ("Salesforce/wikitext", "wikitext"):
        try:
            ds = load_dataset(repo, "wikitext-2-raw-v1", split="test"); break
        except Exception:
            ds = None
    if ds is None:
        raise RuntimeError("could not load wikitext-2-raw-v1 (try: pip install -U datasets)")
    ids = tok("\n\n".join(ds["text"]), return_tensors="pt").input_ids[0]
    k = min(n, ids.shape[0] // seqlen)
    return [ids[i * seqlen:(i + 1) * seqlen] for i in range(k)]

@torch.no_grad()
def perplexity(model, windows):
    dev = next(model.parameters()).device
    nll = ntok = 0
    for w in windows:
        w = w.to(dev)
        out = model(w.unsqueeze(0), labels=w.unsqueeze(0))
        nll += out.loss.item() * (w.numel() - 1); ntok += w.numel() - 1
    return math.exp(nll / ntok)
print("ready.")

In [ ]:
# ======================= CONFIG =======================
MODELS = [
    # OPT scaling ladder (~100x params) -- all fit a 40 GB A100
    "facebook/opt-125m",
    "facebook/opt-1.3b",
    "facebook/opt-2.7b",
    "facebook/opt-6.7b",
    "facebook/opt-13b",       # 26 GB
    # cross-family, open-license, fit 40 GB
    "mistralai/Mistral-7B-v0.1",
    "Qwen/Qwen2.5-7B",
    "Qwen/Qwen2.5-14B",       # 28 GB -- tight but fits; SKIPs if OOM
]
EVAL_WINDOWS = 40
SEQLEN       = 2048
WEIGHTS      = "nvfp4"    # "nvfp4" = W4A4 (real scheme); "fp16" = A-only
SCALE_MODE   = "fp16"     # activation block-scale: "fp16" or "e4m3"
LOAD_8BIT    = False      # set True only to FIT a model too big for the GPU
PURGE_CACHE  = True       # delete each model's HF cache after eval (keeps Colab disk clear)
OUT_CSV      = "codebook_suite_results.csv"   # tiny; set to a Drive path to keep it
SEED         = 0
# ======================================================
import os, shutil, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
global _SCALE_MODE, QUANT_WEIGHTS_NVFP4
_SCALE_MODE = SCALE_MODE
QUANT_WEIGHTS_NVFP4 = (WEIGHTS == "nvfp4")
print(f"weights={WEIGHTS} ({'W4A4' if QUANT_WEIGHTS_NVFP4 else 'A-only'})  purge_cache={PURGE_CACHE}")

def _disk_free_gb():
    try: return shutil.disk_usage("/").free / 1e9
    except Exception: return float("nan")
def _purge(name):
    try:
        from huggingface_hub.constants import HF_HUB_CACHE as base
    except Exception:
        base = os.path.expanduser("~/.cache/huggingface/hub")
    d = os.path.join(base, "models--" + name.replace("/", "--"))
    if os.path.isdir(d): shutil.rmtree(d, ignore_errors=True); print(f"  [purged {d}]")
# fixed codebooks + the two SOLVED-per-model ones (this is where the optimization happens)
EXTRA = ["solved_opt", "solved_pop2"]
COLS  = list(CODEBOOKS) + EXTRA
def _save(rows):
    with open(OUT_CSV, "w", newline="") as f:
        w = csv.writer(f); w.writerow(["model", "fp16"] + COLS + ["solved_ks", "solved_pop2_ks"])
        for name, res in rows:
            w.writerow([name, f"{res['fp16']:.4f}"] + [f"{res[c]:.4f}" for c in COLS]
                       + [res.get("solved_ks", ""), res.get("solved_pop2_ks", "")])

rows = []
for name in MODELS:
    print(f"\n===== {name} =====  (disk free {_disk_free_gb():.0f} GB)", flush=True)
    try:
        tok = AutoTokenizer.from_pretrained(name, use_fast=False)
        kw = dict(device_map="auto", torch_dtype=torch.float16)
        if LOAD_8BIT: kw.update(load_in_8bit=True); kw.pop("torch_dtype")
        model = AutoModelForCausalLM.from_pretrained(name, **kw).eval()
        windows = load_windows(tok, SEQLEN, EVAL_WINDOWS)
        dev = next(model.parameters()).device
        base = perplexity(model, windows)
        install_hooks(model, SEED)
        # SOLVE the constrained-optimal codebook on THIS model's own activations
        mags = collect_mags(model, windows)
        s_lv,  s_ks  = solve_constrained_codebook(mags)
        s_lv2, s_ks2 = solve_constrained_codebook(mags, popcount=2)
        cbs = dict(CODEBOOKS); cbs["solved_opt"] = s_lv; cbs["solved_pop2"] = s_lv2
        res = {"fp16": base, "solved_ks": str(list(s_ks)), "solved_pop2_ks": str(list(s_ks2))}
        for cb, lv in cbs.items():
            set_codebook(lv, dev); res[cb] = perplexity(model, windows)
        print(f"  fp16 {base:.4f}   SOLVED opt={list(s_ks)}  pop2={list(s_ks2)}")
        for cb in COLS:
            print(f"    {cb:12s} PPL {res[cb]:8.4f}  dPPL(exact) {res[cb]-res['exact_GF4']:+7.3f}")
        rows.append((name, res))
        del model; gc.collect(); torch.cuda.empty_cache()
        _save(rows)                          # incremental: survives a later crash
    except Exception as e:
        print(f"  SKIP ({type(e).__name__}: {str(e)[:150]})")
    if PURGE_CACHE: _purge(name)             # free disk before the next (bigger) model

print("\n=== dPPL vs exact_GF4 (negative = beats hand-tuned GF4) ===")
print("model".ljust(26) + "".join(c.rjust(13) for c in COLS))
for name, res in rows:
    print(name.ljust(26) + "".join(f"{res[c]-res['exact_GF4']:+13.3f}" for c in COLS))
print("\n=== solved-per-model codebook (does it match the global {0,2,4,6,8,10,13,16}?) ===")
for name, res in rows:
    print(f"  {name:26s} solved_opt={res['solved_ks']:28s} solved_pop2={res['solved_pop2_ks']}")
print(f"\nwrote {OUT_CSV}")